In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import sklearn

# Check scikit-learn version for OneHotEncoder parameter
sklearn_version = sklearn.__version__
print(f"scikit-learn version: {sklearn_version}")
ohe_params = {'handle_unknown': 'ignore', 'sparse_output': True} if int(sklearn_version.split('.')[1]) >= 2 else {'handle_unknown': 'ignore', 'sparse': True}

df = pd.read_csv('cleaned_data_field.csv')  # Adjust path

# Print dataset size for debugging
print(f"Dataset shape: {df.shape}")

# Define target and features
target = 'value_eur'
X = df.drop(columns=[target, 'long_name', 'release_clause_eur', 'wage_eur', 'overall', 'potential'])
y = df[target]

# Identify categorical and numerical columns
categorical_cols = ['preferred_foot', 'body_type', 'real_face']  # Excluded nationality for speed
numerical_cols = [col for col in X.columns if col not in categorical_cols]

# Handle missing values
for col in numerical_cols:
    X[col] = X[col].fillna(X[col].median())
for col in categorical_cols:
    X[col] = X[col].fillna(X[col].mode()[0])

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(**ohe_params), categorical_cols)
    ])

# Create pipeline
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])

# Split data: 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Print shapes for debugging
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

# Step 4: Cross-Validation on Training Set
cv_r2_scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='r2', n_jobs=-1)
cv_mse_scores = -cross_val_score(pipeline, X_train, y_train, cv=3, scoring='neg_mean_squared_error', n_jobs=-1)

print("\nCross-Validation Performance (3-fold, on Training Set):")
print(f"Average R² Score: {np.mean(cv_r2_scores):.2f} (±{np.std(cv_r2_scores):.2f})")
print(f"Average MSE: {np.mean(cv_mse_scores):.2f} (±{np.std(cv_mse_scores):.2f})\n")

# Step 5: Hyperparameter Tuning with GridSearchCV
param_grid = {
    'regressor__n_estimators': [50, 100],
    'regressor__max_depth': [5, 10],
    'regressor__min_samples_split': [5, 10]
}
grid_search = GridSearchCV(
    pipeline,
    param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    return_train_score=True
)
grid_search.fit(X_train, y_train)

# Record best parameters and scores
print("GridSearchCV Results:")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Cross-Validation R² Score: {grid_search.best_score_:.2f}")
print(f"Mean Training R² Score: {np.mean(grid_search.cv_results_['mean_train_score']):.2f}\n")

# Step 6: Evaluate Best Model on Training and Test Sets
best_model = grid_search.best_estimator_

# Training set evaluation
y_train_pred = best_model.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred)
train_r2 = r2_score(y_train, y_train_pred)

# Test set evaluation
try:
    y_test_pred = best_model.predict(X_test)
    test_mse = mean_squared_error(y_test, y_test_pred)
    test_r2 = r2_score(y_test, y_test_pred)
except Exception as e:
    print(f"Error in test set prediction: {e}")
    raise

# Print performance to check for overfitting
print("Training Set Performance (Best Model):")
print(f"Mean Squared Error: {train_mse:.2f}")
print(f"R² Score: {train_r2:.2f}\n")

print("Testing Set Performance (Best Model):")
print(f"Mean Squared Error: {test_mse:.2f}")
print(f"R² Score: {test_r2:.2f}\n")

# Step 6: Overfitting Check
print("Overfitting Check:")
print(f"Training R² - Test R²: {train_r2 - test_r2:.2f}")
print(f"Training R² - CV R²: {train_r2 - np.mean(cv_r2_scores):.2f}")
if train_r2 - test_r2 > 0.1 or train_r2 - np.mean(cv_r2_scores) > 0.1:
    print("Warning: Potential overfitting detected (large gap between training and test/CV scores).")
else:
    print("No significant overfitting detected.")

# Feature importance
rf_model = best_model.named_steps['regressor']
feature_names = (numerical_cols + 
                 best_model.named_steps['preprocessor']
                 .named_transformers_['cat']
                 .get_feature_names_out(categorical_cols).tolist())
feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)
print(f"\nNumber of features after preprocessing: {len(feature_names)}")
print("Top 5 Feature Importances:")
print(feature_importance_df.head(5))

scikit-learn version: 1.7.0
Dataset shape: (18829, 108)
Training set shape: (15063, 102)
Testing set shape: (3766, 102)

Cross-Validation Performance (3-fold, on Training Set):
Average R² Score: 0.93 (±0.00)
Average MSE: 1118833723299.05 (±108037805733.01)

GridSearchCV Results:
Best Parameters: {'regressor__max_depth': 10, 'regressor__min_samples_split': 5, 'regressor__n_estimators': 100}
Best Cross-Validation R² Score: 0.92
Mean Training R² Score: 0.93

Training Set Performance (Best Model):
Mean Squared Error: 414618271913.70
R² Score: 0.97

Testing Set Performance (Best Model):
Mean Squared Error: 1263758227904.99
R² Score: 0.92

Overfitting Check:
Training R² - Test R²: 0.05
Training R² - CV R²: 0.05
No significant overfitting detected.

Number of features after preprocessing: 106
Top 5 Feature Importances:
               Feature  Importance
27  movement_reactions    0.469501
58                  lm    0.095324
62                  rm    0.092633
0                  age    0.039257
5